# C2-M2 — Execution bridge & G1 signal-parity gate

**Milestone:** C2-M2 (`ARIMA(1,0,0) daily signal feeds the paper account`).
**PRD:** [`.claude/prds/c2-lean-paper.prd.md`](../.claude/prds/c2-lean-paper.prd.md) (scope item 2).
**Platform decision:** [`docs/concepts/lean-setup.md`](../docs/concepts/lean-setup.md) — Alpaca paper (ratified §8.3 fallback).

This is the cross-module E2E notebook (METHODOLOGY §17). It exercises the
full live-inference seam on **real lake fixtures** and renders the **G1 verdict**:

```
get_pit_panel(asof) → forward-return labels → ARIMA(1,0,0) → sign → target position → order
```

**G1 (signal parity):** for every `(symbol, date)`, the bridge's emitted
`target_position` must equal the position the Phase 1 backtest path derives from
the *same* forecast (`np.sign(raw_pred)`), with a pinned **0-mismatch** threshold
(`G1_MAX_MISMATCHES`). The bridge and backtest mappings are coded independently
(`derive_target_position` vs `backtest_path_target_position`); G1 asserts they
agree on real forecasts. The bridge model lives *outside* the engine, so any
mismatch would point at the position-mapping rule — which is pinned once and
shared.

C2-M2 makes **no edge claim** — ARIMA is a placeholder whose P&L is
uninteresting by design. G2 (backtest⇔paper reconciliation) and G3 (≥5-cycle
liveness) are C2-M3.

In [ ]:
import pandas as pd

from quant.execution.lean_bridge import (
    G1_MAX_MISMATCHES,
    AlpacaPaperBridge,
    LeanBridge,
    TargetOrder,
    backtest_path_target_position,
    build_feature_row,
    build_market_order,
    daily_signal,
    derive_target_position,
    plan_order,
    signal_parity_gate_report,
)

# A small slice of the universe + a sweep of as-of dates spanning two years
# (rate-hike 2022–23 and the 2024 rally — ≥2 regimes, the ground for a
# non-trivial parity window). G1 is deterministic; more pairs = stronger.
SYMBOLS = ["SPY", "AAPL", "MSFT", "JPM", "CVX"]
ASOF_DATES = [d.strftime("%Y-%m-%d") for d in pd.date_range("2022-09-30", "2024-09-30", freq="ME")]
print(f"{len(SYMBOLS)} symbols × {len(ASOF_DATES)} as-of dates")
print("G1_MAX_MISMATCHES (pinned):", G1_MAX_MISMATCHES)

## Step 1 — emit a daily signal

`daily_signal(asof)` reads the point-in-time panel, fits ARIMA(1,0,0) on each
symbol's forward-return label series, forecasts one step, and maps the sign to a
target position. The reader returns only bars with `timestamp ≤ asof`, so the
forecast carries no look-ahead.

In [ ]:
one = daily_signal(ASOF_DATES[-1], SYMBOLS)
pd.DataFrame(
    [
        {"symbol": s.symbol, "asof": s.asof.date(), "forecast": s.forecast, "target_position": s.target_position}
        for s in one.values()
    ]
).set_index("symbol")

## Step 2 — G1 signal-parity gate (the merge-blocking verdict)

Sweep every `(symbol, as-of)` pair. For each emitted forecast, build the
`(bridge, backtest)` target-position pair and run the gate. PASS requires **zero**
material mismatches over a non-empty window.

In [ ]:
checks = []
rows = []
for asof in ASOF_DATES:
    for sym, sig in daily_signal(asof, SYMBOLS).items():
        bridge_tp = derive_target_position(sig.forecast)
        backtest_tp = backtest_path_target_position(sig.forecast)
        checks.append((bridge_tp, backtest_tp))
        rows.append({"asof": asof, "symbol": sym, "forecast": sig.forecast, "bridge": bridge_tp, "backtest": backtest_tp})

g1 = signal_parity_gate_report(checks)
print(f"G1 checked {g1.n_checked} (symbol, date) pairs — {g1.n_mismatches} mismatches")
print("G1 VERDICT:", "PASS ✅" if g1.passed else "FAIL ❌")
pd.DataFrame(rows).tail(8)

## Step 3 — reader → features seam (the future-model hook)

ARIMA forecasts from labels, but the deployment path also exposes
`build_feature_row` — the `get_pit_panel → build_features(asof)` seam a future
feature-consuming B-model plugs into. The same `asof` truncation gives the
21-column same-day feature matrix (C1-M2 G2 parity).

In [ ]:
feat = build_feature_row(SYMBOLS, ASOF_DATES[-1])
print({sym: df.shape for sym, df in feat.items()})
feat["SPY"].tail(3)

## Step 4 — the broker boundary (`ExecutionBridge`)

A target position becomes an order through `plan_order` (pure, signed-delta) and
`build_market_order`. The `AlpacaPaperBridge` then submits it (the live order path
was already proven in C2-M1's hello-world). `LeanBridge` is the deferred swap.

In [ ]:
# Pure order planning: flat → long, long → flat, flat → short, cross short→long.
examples = [
    ("flat->long", plan_order("SPY", 1, 0.0)),
    ("long->flat", plan_order("SPY", 0, 1.0)),
    ("flat->short", plan_order("SPY", -1, 0.0)),
    ("short->long", plan_order("SPY", 1, -1.0)),
    ("on-target", plan_order("SPY", 1, 1.0)),
]
for label, intent in examples:
    print(f"{label:12s} -> {intent}")

req = build_market_order(plan_order("SPY", 1, 0.0))
print("\nMarketOrderRequest:", req.symbol, req.qty, req.side, req.time_in_force)
print("LeanBridge deferred:", end=" ")
try:
    LeanBridge()
except NotImplementedError as exc:
    print(type(exc).__name__)

In [ ]:
# Live paper submission is guarded: it requires .env credentials and is the
# operational concern of the C2-M3 >=5-cycle loop. The boot+order path itself is
# proven by C2-M1 (docs/concepts/lean-setup.md section 2). Flip SUBMIT to True to
# place one real paper order from this notebook.
SUBMIT = False
if SUBMIT:
    bridge = AlpacaPaperBridge.from_settings()
    print("account:", bridge.account_summary())
    target_sym, target_sig = next(iter(one.items()))
    print("placing:", bridge.place_target(TargetOrder(target_sym, target_sig.target_position)))
else:
    print("SUBMIT=False - live paper order skipped (C2-M1 already proved the order path).")

## Verdict

- **G1 (signal parity):** the bridge emits the backtest's decision on every
  `(symbol, date)` over the sweep — 0 mismatches, the pinned threshold (Step 2).
- The reader→features→ARIMA→sign→order path runs end-to-end on real lake data.
- The `ExecutionBridge` boundary (Alpaca primary, LEAN deferred) is the contract
  C3/C4 consume.

**Next:** C2-M3 ships the G2 reconciliation harness (paper⇔backtest ≤1% with a
decomposed residual) and the G3 ≥5-cycle liveness runbook.